# Retrieval Tuning Notebook

Exploratory notebook for tuning the retriever's `top_k` parameter and inspecting retrieved chunks for sample applicant profiles. This is a development-time exploration aid; it is not part of the production pipeline or API.

In [1]:
%pip install --upgrade pandas pyarrow faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip show sentence-transformers, faiss-cpu, pandas, scikit-learn, matplotlib, seaborn, jupyterlab

In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.infrastructure.embedder import Embedder
from src.infrastructure.vector_store import VectorStore
from src.model import Retriever
from src.domain.entities import UserProfile

embedder = Embedder()
vector_store = VectorStore("../artifacts/faiss.index", "../artifacts/chunks.parquet").load()
print("Loaded", len(vector_store.chunks), "chunks")

c:\My Dev\Upgrad\mydev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 25 chunks


In [5]:
profile = UserProfile(
    annual_income=800_000,
    age=27,
    cibil=760,
    employment="salaried",
    existing_relationship=False,
    banks=["SBI", "HDFC"],
)

for k in [2, 4, 6, 8, 12]:
    retriever = Retriever(embedder, vector_store, top_k=k)
    retrieved = retriever.retrieve(profile.to_query())
    print(f"top_k={k}: retrieved {len(retrieved)} chunks")
    for chunk in retrieved:
        print("  ", chunk["chunk_id"], chunk["bank"], chunk["card"], round(chunk["score"], 3))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5397.06it/s]


top_k=2: retrieved 2 chunks
   hdfc_cards#1 HDFC HDFC MoneyBack+ 0.718
   sbi_cards#8 SBI SBI Prime 0.701
top_k=4: retrieved 4 chunks
   hdfc_cards#1 HDFC HDFC MoneyBack+ 0.718
   sbi_cards#8 SBI SBI Prime 0.701
   sbi_cards#5 SBI SBI SimplySAVE 0.693
   sbi_cards#4 SBI SBI SimplySAVE 0.681
top_k=6: retrieved 6 chunks
   hdfc_cards#1 HDFC HDFC MoneyBack+ 0.718
   sbi_cards#8 SBI SBI Prime 0.701
   sbi_cards#5 SBI SBI SimplySAVE 0.693
   sbi_cards#4 SBI SBI SimplySAVE 0.681
   sbi_cards#1 SBI SBI SimplyCLICK 0.671
   sbi_cards#11 SBI SBI Elite 0.658
top_k=8: retrieved 8 chunks
   hdfc_cards#1 HDFC HDFC MoneyBack+ 0.718
   sbi_cards#8 SBI SBI Prime 0.701
   sbi_cards#5 SBI SBI SimplySAVE 0.693
   sbi_cards#4 SBI SBI SimplySAVE 0.681
   sbi_cards#1 SBI SBI SimplyCLICK 0.671
   sbi_cards#11 SBI SBI Elite 0.658
   hdfc_cards#4 HDFC HDFC Millennia 0.657
   sbi_cards#10 SBI SBI Elite 0.62
top_k=12: retrieved 12 chunks
   hdfc_cards#1 HDFC HDFC MoneyBack+ 0.718
   sbi_cards#8 SBI SBI Prime 0.7

## Notes

* Increasing `top_k` trades off precision (more irrelevant chunks may be included) against recall (more of the relevant rules for a multi-hop decision are surfaced).
* See `evaluation/run_ablation.py::a2_top_k_sweep` for the quantitative, gold-set-driven version of this exploration (A2, one of the two strongest novel contributions in the ablation matrix).